# PaMCP MultiBenchmark v1 — Colab runner

This notebook acquires pinned public benchmark snapshots, normalizes MCP-Atlas and ToolRet with discovery/evaluator separation, runs the ToolRet BM25 retrieval baseline, computes task-level ranking metrics, and exports reproducible artifacts. It does **not** claim PaMCP superiority or use MCP-Atlas trajectories as exhaustive tool-set ground truth. Full MCP server execution is a separate sandbox workflow.

In [ ]:
import os, subprocess, json, pathlib, hashlib, re, sys
REPO = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
BRANCH = 'research/pamcp-multibenchmark-v1'
if not pathlib.Path('/content/taas-provenance-discovery').exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO,'/content/taas-provenance-discovery'], check=True)
os.chdir('/content/taas-provenance-discovery')
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[experiment,benchmark-hf,test]'], check=True)
print('Repository:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',
 'tests/test_pamcp_benchmark_io.py',
 'tests/test_pamcp_benchmark_adapters.py',
 'tests/test_mcp_atlas_adapter.py',
 'tests/test_toolret_adapter.py',
 'tests/test_evaluation_metrics.py',
 'tests/test_declared_effects_are_not_observations.py'], check=True)

## 1. Acquire MCP-Atlas public dataset

The public release is 500 tasks. The full benchmark contains additional private tasks that are not acquired here. The dataset card declares CC-BY-4.0; preserve attribution and pin the exact dataset revision.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from taas.experiments.mcp_atlas_adapter import adapt_mcp_atlas_parquet
api = HfApi()
atlas_info = api.dataset_info('ScaleAI/MCP-Atlas')
atlas_revision = atlas_info.sha
atlas_files = api.list_repo_files('ScaleAI/MCP-Atlas', repo_type='dataset', revision=atlas_revision)
parquet_files = [x for x in atlas_files if x.endswith('.parquet')]
assert len(parquet_files) == 1, f'Expected one public parquet file, found {parquet_files}'
atlas_path = hf_hub_download('ScaleAI/MCP-Atlas', parquet_files[0], repo_type='dataset', revision=atlas_revision)
atlas_manifest = adapt_mcp_atlas_parquet(atlas_path, 'data/processed/mcp_atlas_public')
atlas_manifest['source_revision'] = atlas_revision
pathlib.Path('data/processed/mcp_atlas_public/manifest.json').write_text(json.dumps(atlas_manifest, indent=2))
print(json.dumps(atlas_manifest['counts'], indent=2))

## 2. Acquire ToolRet at independent immutable revisions

ToolRet stores queries and tools in separate Hugging Face repositories. Their revisions are pinned independently. The complete tool corpus is mandatory; deriving candidates from positive qrels alone would invalidate retrieval evaluation.

In [ ]:
from taas.experiments.toolret_adapter import adapt_toolret_hf
query_info = api.dataset_info('mangopy/ToolRet-Queries')
tool_info = api.dataset_info('mangopy/ToolRet-Tools')
toolret_manifest = adapt_toolret_hf(
    'data/processed/toolret',
    query_revision=query_info.sha,
    tool_revision=tool_info.sha,
)
print(json.dumps(toolret_manifest['counts'], indent=2))
print('Query revision:', query_info.sha)
print('Tool revision:', tool_info.sha)

## 3. Run ToolRet BM25 baseline

This is a heterogeneous retrieval baseline, not an MCP execution result. It uses the full ToolRet candidate corpus and original graded relevance judgments. The ranking metric evaluator penalizes abstentions as empty rankings.

In [ ]:
from rank_bm25 import BM25Okapi
from taas.experiments.evaluation_metrics import ranking_metrics
tools = json.loads(pathlib.Path('data/processed/toolret/discovery/tools.json').read_text())
tasks = json.loads(pathlib.Path('data/processed/toolret/discovery/tasks.json').read_text())
refs = json.loads(pathlib.Path('data/processed/toolret/evaluator/task_references.json').read_text())
def tok(s): return re.findall(r'[a-z0-9_]+', str(s).lower())
tool_ids = [t['tool_id'] for t in tools]
tool_docs = [tok(' '.join([t.get('name',''), t.get('description',''), json.dumps(t.get('input_schema',{}), sort_keys=True)])) for t in tools]
bm25 = BM25Okapi(tool_docs)
qrels = {r['task_id']:{x['tool_id']:float(x.get('relevance',1)) for x in r['relevance_judgments']} for r in refs}
rankings = {}
for i, task in enumerate(tasks, start=1):
    scores = bm25.get_scores(tok(task['prompt']))
    order = sorted(range(len(scores)), key=lambda j: (-float(scores[j]), tool_ids[j]))[:10]
    rankings[task['task_id']] = [tool_ids[j] for j in order]
    if i % 500 == 0: print(f'Ranked {i}/{len(tasks)} tasks')
metrics = ranking_metrics(rankings, qrels, ks=(1,5,10))
out = pathlib.Path('artifacts/multibenchmark/toolret_bm25')
out.mkdir(parents=True, exist_ok=True)
(out/'rankings.json').write_text(json.dumps(rankings))
(out/'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

In [ ]:
import matplotlib.pyplot as plt
ks = [1,5,10]
ndcg = [metrics['metrics'][str(k)][f'ndcg@{k}']['value'] for k in ks]
hit = [metrics['metrics'][str(k)][f'hit@{k}']['value'] for k in ks]
plt.figure(figsize=(7,4)); plt.plot(ks, ndcg, marker='o', label='nDCG@k'); plt.plot(ks, hit, marker='s', label='Hit@k')
plt.xlabel('k'); plt.ylabel('Macro query-level score'); plt.title('ToolRet — BM25 retrieval baseline'); plt.grid(True); plt.legend(); plt.show()
pool_sizes = [len(x['enabled_tool_ids']) for x in json.loads(pathlib.Path('data/processed/mcp_atlas_public/discovery/task_tool_visibility.json').read_text())]
plt.figure(figsize=(7,4)); plt.hist(pool_sizes, bins=range(min(pool_sizes), max(pool_sizes)+2)); plt.xlabel('Enabled tools per MCP-Atlas task'); plt.ylabel('Task count'); plt.title('MCP-Atlas candidate-pool distribution'); plt.show()

## 4. Reproducibility and export

Keep the raw source revisions, generated manifests, per-task rankings, metrics and plots together. Do not publish only aggregate scores. The MCP-Atlas claims and trajectories remain evaluator-side; this notebook does not convert the successful trajectory into an exhaustive set of acceptable toolchains.

In [ ]:
import zipfile
artifact_root = pathlib.Path('artifacts/multibenchmark')
artifact_root.mkdir(parents=True, exist_ok=True)
summary = {
  'repository_commit': subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip(),
  'mcp_atlas_revision': atlas_revision,
  'toolret_query_revision': query_info.sha,
  'toolret_tool_revision': tool_info.sha,
  'toolret_metrics': metrics,
  'scientific_status': 'retrieval baseline executed; no PaMCP superiority claim; E3 effect oracle not supplied'
}
(artifact_root/'run_summary.json').write_text(json.dumps(summary, indent=2))
zip_path = '/content/PaMCP_MultiBenchmark_v1_artifacts.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
  for p in pathlib.Path('data/processed').rglob('*'):
    if p.is_file(): z.write(p, p.relative_to('.'))
  for p in artifact_root.rglob('*'):
    if p.is_file(): z.write(p, p.relative_to('.'))
print('Artifact bundle:', zip_path)